# 03 — Normalização, criação de colunas e universidades

Este notebook lê os arquivos enriquecidos do passo 02 e cria o corpus limpo.

## Melhorias desta versão

1. Encontra automaticamente a raiz do projeto, mesmo quando o notebook está dentro de `Scielo/03.data_cleaning/`.
2. Procura arquivos enriquecidos em múltiplos padrões e diretórios.
3. Ignora CSV vazio, sem colunas ou sem linhas, salvando log dos arquivos pulados.
4. Normaliza HTML, listas, anos, universidades/instituições, estados e regiões.
5. Filtra o período do projeto (`1970–2025`).
6. Corrige tipo de documento com regra conservadora.
7. Classifica campo de estudo e metodologia em V1 heurística.
8. Salva arquivos técnicos, com listas em JSON, e arquivos amigáveis para Google Sheets.

## Saída padrão

```text
data/03_clean/
├── 03_corpus_<keyword_slug>_scielo_clean.csv
├── 03_corpus_<keyword_slug>_scielo_clean_sheets.csv
├── 03_corpus_scielo_clean_combined.csv
└── 03_corpus_scielo_clean_combined_sheets.csv

logs/
└── skipped_normalize_files.csv
```

In [1]:
# Instalação, caso precise:
# !pip install beautifulsoup4 pandas tqdm unidecode openpyxl

In [2]:
import ast
import html
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError
from bs4 import BeautifulSoup
from tqdm import tqdm
from unidecode import unidecode

## 1. Configuração geral

In [3]:
def find_project_dir(start=None):
    """Encontra a raiz do projeto subindo diretórios a partir do notebook.

    Critérios:
    - pasta contendo `Scielo/`; e
    - algum marcador de projeto (`README.md`, `.gitignore` ou `requirements.txt`).

    Isso permite rodar o notebook de dentro de:
    `Scielo/03.data_cleaning/`, `Scielo/04.data_analysis/` ou da raiz.
    """
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        has_scielo_folder = (candidate / 'Scielo').exists()
        has_project_marker = any(
            (candidate / marker).exists()
            for marker in ['README.md', '.gitignore', 'requirements.txt']
        )

        if has_scielo_folder and has_project_marker:
            return candidate

    # fallback: assume pasta atual
    return current


PROJECT_DIR = find_project_dir()

DATA_DIR = PROJECT_DIR / 'data'
LOG_DIR = PROJECT_DIR / 'logs'

ENRICHED_DIR = DATA_DIR / '02_enriched'
CLEAN_DIR = DATA_DIR / '03_clean'
CLEAN_SHEETS_DIR = CLEAN_DIR

for path in [DATA_DIR, LOG_DIR, ENRICHED_DIR, CLEAN_DIR, CLEAN_SHEETS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

START_YEAR = 1970
END_YEAR = 2025

# Use None para processar todos os arquivos enriquecidos.
# Exemplo: KEYWORD_SLUGS = ['elites', 'sociologia_das_elites']
KEYWORD_SLUGS = None

print('PROJECT_DIR:', PROJECT_DIR)
print('ENRICHED_DIR:', ENRICHED_DIR)
print('CLEAN_DIR:', CLEAN_DIR)

PROJECT_DIR: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos
ENRICHED_DIR: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/02_enriched
CLEAN_DIR: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean


## 2. Funções de limpeza

In [4]:
def clean_text(value):
    """Remove HTML, decodifica entidades e normaliza espaços."""
    if value is None or pd.isna(value):
        return ''

    value = str(value)
    value = BeautifulSoup(value, 'html.parser').get_text(' ', strip=True)
    value = html.unescape(value)
    value = re.sub(r'\s+', ' ', value).strip()
    return value


def normalize_key(value):
    """Normaliza string para comparação robusta."""
    value = clean_text(value).lower()
    value = unidecode(value)
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    value = re.sub(r'\s+', ' ', value).strip()
    return value


def safe_literal_list(value):
    """Converte string de lista ou JSON para lista Python."""
    if isinstance(value, list):
        return [clean_text(x) for x in value if clean_text(x)]

    if value is None or pd.isna(value):
        return []

    value = str(value).strip()

    if value in ['', '[]', 'nan', 'None', '<NA>']:
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [clean_text(x) for x in parsed if clean_text(x)]
        return [clean_text(parsed)] if clean_text(parsed) else []
    except Exception:
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return [clean_text(x) for x in parsed if clean_text(x)]
        except Exception:
            pass

        return [clean_text(value)] if clean_text(value) else []


def extract_year(value):
    """Extrai ano de uma string."""
    value = clean_text(value)
    match = re.search(r'\b(19\d{2}|20\d{2})\b', value)
    return int(match.group(1)) if match else pd.NA


def serialize_list_columns(df, list_columns):
    """Serializa listas como JSON para CSV técnico estável."""
    out = df.copy()
    for col in list_columns:
        if col in out.columns:
            out[col] = out[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
            )
    return out


def list_to_sheet_cell(value, sep=' | '):
    """Converte listas em texto simples para Google Sheets."""
    if isinstance(value, list):
        return sep.join(clean_text(x) for x in value if clean_text(x))

    if value is None or pd.isna(value):
        return ''

    return clean_text(value)


def get_series(df, col, default=''):
    """Retorna uma coluna existente ou uma Series padrão do mesmo tamanho."""
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def load_csv_safe(path, required_any=None):
    """Carrega CSV de forma robusta.

    Retorna:
    - DataFrame, se o arquivo for válido;
    - None, se estiver vazio, sem colunas, sem linhas ou sem colunas mínimas.
    """
    path = Path(path)

    if not path.exists():
        return None, 'arquivo inexistente'

    if path.stat().st_size == 0:
        return None, 'arquivo vazio'

    try:
        df = pd.read_csv(path)
    except EmptyDataError:
        return None, 'CSV sem colunas'
    except Exception as exc:
        return None, f'erro de leitura: {exc}'

    if df.empty:
        return None, 'CSV sem linhas'

    if required_any:
        if not any(col in df.columns for col in required_any):
            return None, f'nenhuma coluna obrigatória encontrada: {required_any}'

    return df, ''


def get_keyword_slug_from_enriched_file(path):
    """Extrai keyword_slug a partir de nomes de arquivos enriquecidos."""
    name = Path(path).name
    slug = name

    replacements = [
        ('02_scielo_articles_', ''),
        ('scielo_', ''),
        ('enriched_', ''),
        ('_enriched.csv', ''),
        ('_articles_enriched.csv', ''),
        ('.csv', ''),
    ]

    for old, new in replacements:
        slug = slug.replace(old, new)

    return slug.strip('_')


def create_google_sheets_friendly_df(df):
    """Cria versão achatada e legível para análise em Google Sheets.

    A base técnica preserva listas em JSON; esta versão transforma listas em texto separado por ` | `
    e renomeia colunas para nomes curtos em português.
    """
    if df.empty:
        return pd.DataFrame()

    out = pd.DataFrame(index=df.index)

    out['termo_busca'] = get_series(df, 'search_keyword')
    out['grupo_termo'] = get_series(df, 'search_group')
    out['slug_termo_busca'] = get_series(df, 'search_keyword_slug')
    out['titulo'] = get_series(df, 'article_title')
    out['autores'] = get_series(df, 'article_authors').apply(list_to_sheet_cell)
    out['periodico'] = get_series(df, 'article_journal')
    out['ano'] = get_series(df, 'article_year')
    out['doi'] = get_series(df, 'article_doi')
    out['idioma'] = get_series(df, 'article_language')
    out['resumo'] = get_series(df, 'article_abstract')
    out['palavras_chave'] = get_series(df, 'article_keywords').apply(list_to_sheet_cell)
    out['instituicoes_brutas'] = get_series(df, 'institutions_raw_combined').apply(list_to_sheet_cell)
    out['instituicoes_normalizadas'] = get_series(df, 'institutions_normalized').apply(list_to_sheet_cell)
    out['instituicao_principal'] = get_series(df, 'main_institution')
    out['estados'] = get_series(df, 'institution_states').apply(list_to_sheet_cell)
    out['regioes'] = get_series(df, 'institution_regions').apply(list_to_sheet_cell)
    out['paises'] = get_series(df, 'institution_countries').apply(list_to_sheet_cell)
    out['estado_principal'] = get_series(df, 'main_state')
    out['regiao_principal'] = get_series(df, 'main_region')
    out['pais_principal'] = get_series(df, 'main_country')
    out['campos_estudo'] = get_series(df, 'fields_detected').apply(list_to_sheet_cell)
    out['metodologias'] = get_series(df, 'methodologies_detected').apply(list_to_sheet_cell)
    out['tipo_documento'] = get_series(df, 'document_type_v2')
    out['eh_artigo_pesquisa'] = get_series(df, 'is_core_research_article_v2')
    out['qtd_referencias'] = get_series(df, 'article_references_count')
    out['link'] = get_series(df, 'link')
    out['arquivo_origem'] = get_series(df, 'source_file')

    return out

## 3. Catálogo canônico de universidades e instituições

Edite este catálogo conforme novas instituições aparecerem. A lógica é:

- `canonical`: nome padronizado
- `state`: UF
- `region`: região brasileira
- `country`: país
- `patterns`: variações e siglas que podem aparecer no texto

A normalização não deve ser tratada como verdade absoluta. Ela é uma V1 auditável.

In [5]:
STATE_REGION_MAP = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
}

INSTITUTION_CATALOG = [
    {'canonical': 'Universidade de São Paulo', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Universidade de São Paulo', 'University of São Paulo', 'University of Sao Paulo', 'USP']},
    {'canonical': 'Universidade Estadual de Campinas', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Universidade Estadual de Campinas', 'University of Campinas', 'UNICAMP']},
    {'canonical': 'Universidade Estadual Paulista', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Universidade Estadual Paulista', 'UNESP']},
    {'canonical': 'Universidade Federal de São Carlos', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Universidade Federal de São Carlos', 'Universidade Federal de Sao Carlos', 'UFSCar']},
    {'canonical': 'Pontifícia Universidade Católica de São Paulo', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Pontifícia Universidade Católica de São Paulo', 'Pontificia Universidade Catolica de Sao Paulo', 'PUC-SP', 'PUC SP']},
    {'canonical': 'Universidade Presbiteriana Mackenzie', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Universidade Presbiteriana Mackenzie', 'Mackenzie']},
    {'canonical': 'Fundação Getulio Vargas', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Fundação Getulio Vargas', 'Fundação Getúlio Vargas', 'Fundacao Getulio Vargas', 'FGV', 'CPDOC']},
    {'canonical': 'Centro Brasileiro de Análise e Planejamento', 'state': 'SP', 'country': 'Brasil', 'patterns': ['Centro Brasileiro de Análise e Planejamento', 'Centro Brasileiro de Analise e Planejamento', 'CEBRAP']},

    {'canonical': 'Universidade Federal do Rio de Janeiro', 'state': 'RJ', 'country': 'Brasil', 'patterns': ['Universidade Federal do Rio de Janeiro', 'Federal University of Rio de Janeiro', 'UFRJ']},
    {'canonical': 'Universidade do Estado do Rio de Janeiro', 'state': 'RJ', 'country': 'Brasil', 'patterns': ['Universidade do Estado do Rio de Janeiro', 'UERJ', 'IESP-UERJ', 'IESP']},
    {'canonical': 'Universidade Federal Fluminense', 'state': 'RJ', 'country': 'Brasil', 'patterns': ['Universidade Federal Fluminense', 'UFF']},
    {'canonical': 'Pontifícia Universidade Católica do Rio de Janeiro', 'state': 'RJ', 'country': 'Brasil', 'patterns': ['Pontifícia Universidade Católica do Rio de Janeiro', 'Pontificia Universidade Catolica do Rio de Janeiro', 'PUC-Rio', 'PUC Rio']},
    {'canonical': 'Fundação Oswaldo Cruz', 'state': 'RJ', 'country': 'Brasil', 'patterns': ['Fundação Oswaldo Cruz', 'Fundacao Oswaldo Cruz', 'Fiocruz', 'FIOCRUZ']},
    {'canonical': 'Fundação Casa de Rui Barbosa', 'state': 'RJ', 'country': 'Brasil', 'patterns': ['Fundação Casa de Rui Barbosa', 'Fundacao Casa de Rui Barbosa']},

    {'canonical': 'Universidade Federal de Minas Gerais', 'state': 'MG', 'country': 'Brasil', 'patterns': ['Universidade Federal de Minas Gerais', 'Federal University of Minas Gerais', 'UFMG']},
    {'canonical': 'Universidade Federal de Juiz de Fora', 'state': 'MG', 'country': 'Brasil', 'patterns': ['Universidade Federal de Juiz de Fora', 'UFJF']},
    {'canonical': 'Universidade Federal de Uberlândia', 'state': 'MG', 'country': 'Brasil', 'patterns': ['Universidade Federal de Uberlândia', 'Universidade Federal de Uberlandia', 'UFU']},

    {'canonical': 'Universidade Federal do Rio Grande do Sul', 'state': 'RS', 'country': 'Brasil', 'patterns': ['Universidade Federal do Rio Grande do Sul', 'Federal University of Rio Grande do Sul', 'UFRGS']},
    {'canonical': 'Pontifícia Universidade Católica do Rio Grande do Sul', 'state': 'RS', 'country': 'Brasil', 'patterns': ['Pontifícia Universidade Católica do Rio Grande do Sul', 'Pontificia Universidade Catolica do Rio Grande do Sul', 'PUCRS', 'PUC-RS']},
    {'canonical': 'Universidade do Vale do Rio dos Sinos', 'state': 'RS', 'country': 'Brasil', 'patterns': ['Universidade do Vale do Rio dos Sinos', 'UNISINOS']},

    {'canonical': 'Universidade Federal do Paraná', 'state': 'PR', 'country': 'Brasil', 'patterns': ['Universidade Federal do Paraná', 'Universidade Federal do Parana', 'UFPR']},
    {'canonical': 'Universidade Estadual de Maringá', 'state': 'PR', 'country': 'Brasil', 'patterns': ['Universidade Estadual de Maringá', 'Universidade Estadual de Maringa', 'UEM']},
    {'canonical': 'Universidade Estadual de Londrina', 'state': 'PR', 'country': 'Brasil', 'patterns': ['Universidade Estadual de Londrina', 'UEL']},

    {'canonical': 'Universidade Federal de Santa Catarina', 'state': 'SC', 'country': 'Brasil', 'patterns': ['Universidade Federal de Santa Catarina', 'UFSC']},
    {'canonical': 'Universidade do Estado de Santa Catarina', 'state': 'SC', 'country': 'Brasil', 'patterns': ['Universidade do Estado de Santa Catarina', 'UDESC']},

    {'canonical': 'Universidade de Brasília', 'state': 'DF', 'country': 'Brasil', 'patterns': ['Universidade de Brasília', 'Universidade de Brasilia', 'University of Brasília', 'University of Brasilia', 'UnB', 'UNB']},
    {'canonical': 'Universidade Federal de Goiás', 'state': 'GO', 'country': 'Brasil', 'patterns': ['Universidade Federal de Goiás', 'Universidade Federal de Goias', 'UFG']},
    {'canonical': 'Universidade Federal de Mato Grosso', 'state': 'MT', 'country': 'Brasil', 'patterns': ['Universidade Federal de Mato Grosso', 'UFMT']},
    {'canonical': 'Universidade Federal de Mato Grosso do Sul', 'state': 'MS', 'country': 'Brasil', 'patterns': ['Universidade Federal de Mato Grosso do Sul', 'UFMS']},

    {'canonical': 'Universidade Federal da Bahia', 'state': 'BA', 'country': 'Brasil', 'patterns': ['Universidade Federal da Bahia', 'Federal University of Bahia', 'UFBA']},
    {'canonical': 'Universidade Federal de Pernambuco', 'state': 'PE', 'country': 'Brasil', 'patterns': ['Universidade Federal de Pernambuco', 'UFPE']},
    {'canonical': 'Universidade Federal do Ceará', 'state': 'CE', 'country': 'Brasil', 'patterns': ['Universidade Federal do Ceará', 'Universidade Federal do Ceara', 'UFC']},
    {'canonical': 'Universidade Federal do Rio Grande do Norte', 'state': 'RN', 'country': 'Brasil', 'patterns': ['Universidade Federal do Rio Grande do Norte', 'UFRN']},
    {'canonical': 'Universidade Federal da Paraíba', 'state': 'PB', 'country': 'Brasil', 'patterns': ['Universidade Federal da Paraíba', 'Universidade Federal da Paraiba', 'UFPB']},
    {'canonical': 'Universidade Federal de Alagoas', 'state': 'AL', 'country': 'Brasil', 'patterns': ['Universidade Federal de Alagoas', 'UFAL']},
    {'canonical': 'Universidade Federal de Sergipe', 'state': 'SE', 'country': 'Brasil', 'patterns': ['Universidade Federal de Sergipe', 'UFS']},
    {'canonical': 'Universidade Federal do Maranhão', 'state': 'MA', 'country': 'Brasil', 'patterns': ['Universidade Federal do Maranhão', 'Universidade Federal do Maranhao', 'UFMA']},
    {'canonical': 'Universidade Federal do Piauí', 'state': 'PI', 'country': 'Brasil', 'patterns': ['Universidade Federal do Piauí', 'Universidade Federal do Piaui', 'UFPI']},

    {'canonical': 'Universidade Federal do Pará', 'state': 'PA', 'country': 'Brasil', 'patterns': ['Universidade Federal do Pará', 'Universidade Federal do Para', 'UFPA']},
    {'canonical': 'Universidade Federal do Amazonas', 'state': 'AM', 'country': 'Brasil', 'patterns': ['Universidade Federal do Amazonas', 'UFAM']},
    {'canonical': 'Universidade Federal do Acre', 'state': 'AC', 'country': 'Brasil', 'patterns': ['Universidade Federal do Acre', 'UFAC']},
    {'canonical': 'Universidade Federal de Rondônia', 'state': 'RO', 'country': 'Brasil', 'patterns': ['Universidade Federal de Rondônia', 'Universidade Federal de Rondonia', 'UNIR']},

    # Instituições estrangeiras recorrentes em coautoria podem ser adicionadas aqui.
    {'canonical': 'University of Oxford', 'state': '', 'country': 'Reino Unido', 'patterns': ['University of Oxford', 'Oxford University']},
    {'canonical': 'Harvard University', 'state': '', 'country': 'Estados Unidos', 'patterns': ['Harvard University']},
    {'canonical': 'Sciences Po', 'state': '', 'country': 'França', 'patterns': ['Sciences Po', 'Institut d’études politiques de Paris', 'Institut d etudes politiques de Paris']},
]

for item in INSTITUTION_CATALOG:
    item['region'] = STATE_REGION_MAP.get(item.get('state', ''), '')
    item['pattern_keys'] = [normalize_key(pattern) for pattern in item['patterns']]

## 4. Funções de normalização institucional

In [6]:
# Configurações de performance para normalização institucional.
# O gargalo anterior estava em varrer article_full_text_sample grande com regex para cada linha.
# Agora usamos: texto curto, padrões pré-processados e busca por substring com fronteiras via espaços.
INSTITUTION_TEXT_DETECTION_ENABLED = True
INSTITUTION_TEXT_MAX_CHARS = 2500

# Se True, tenta detectar instituições no começo do texto integral do artigo.
# Mantenha False para rodadas grandes; ligue apenas se a cobertura institucional ficar ruim.
USE_FULL_TEXT_FOR_INSTITUTION_DETECTION = False


def _build_institution_lookup(catalog):
    """Prepara catálogo para matching rápido, sem recompilar regex por linha."""
    lookup = []

    for item in catalog:
        for pattern_key in item.get('pattern_keys', []):
            pattern_key = normalize_key(pattern_key)

            if not pattern_key:
                continue

            lookup.append({
                'pattern_key': pattern_key,
                'pattern_padded': f' {pattern_key} ',
                'canonical': item.get('canonical', ''),
                'item': item,
            })

    # Padrões mais longos primeiro: reduz risco de match genérico antes de match específico.
    lookup = sorted(
        lookup,
        key=lambda x: len(x['pattern_key']),
        reverse=True,
    )

    return lookup


INSTITUTION_LOOKUP = _build_institution_lookup(INSTITUTION_CATALOG)


def normalize_institution_value(value):
    """Normaliza uma instituição isolada para o catálogo canônico."""
    value_clean = clean_text(value)
    value_key = normalize_key(value_clean)

    if not value_key:
        return None

    value_padded = f' {value_key} '

    for lookup in INSTITUTION_LOOKUP:
        pattern_key = lookup['pattern_key']

        # Para siglas e nomes curtos, exige fronteira por espaço.
        # Para nomes longos, substring normalizada é suficiente.
        if len(pattern_key) <= 5:
            matched = lookup['pattern_padded'] in value_padded
        else:
            matched = pattern_key in value_key

        if matched:
            return lookup['item']

    return {
        'canonical': value_clean,
        'state': '',
        'region': '',
        'country': '',
        'patterns': [],
        'pattern_keys': [],
    }


def detect_institutions_from_text(text):
    """Detecta instituições canônicas no texto livre de forma rápida.

    Evita regex por linha. Primeiro normaliza o texto, depois usa busca por substring.
    """
    if not INSTITUTION_TEXT_DETECTION_ENABLED:
        return []

    text_key = normalize_key(text)

    if not text_key:
        return []

    text_padded = f' {text_key} '
    found = []
    seen = set()

    for lookup in INSTITUTION_LOOKUP:
        item = lookup['item']
        canonical = item.get('canonical', '')

        if canonical in seen:
            continue

        pattern_key = lookup['pattern_key']

        if len(pattern_key) <= 5:
            matched = lookup['pattern_padded'] in text_padded
        else:
            matched = pattern_key in text_key

        if matched:
            found.append(item)
            seen.add(canonical)

    return found


def build_text_for_institution_detection(row):
    """Monta texto curto para detecção institucional.

    O texto integral é caro e geralmente desnecessário. Instituições costumam aparecer no topo,
    nas afiliações, publisher, título/resumo ou metadados iniciais.
    """
    parts = [
        row.get('article_title', ''),
        row.get('article_publisher', ''),
        row.get('article_abstract', ''),
    ]

    if USE_FULL_TEXT_FOR_INSTITUTION_DETECTION:
        parts.append(str(row.get('article_full_text_sample', ''))[:INSTITUTION_TEXT_MAX_CHARS])

    return ' '.join(clean_text(part) for part in parts if clean_text(part))


def normalize_institutions_for_row(row):
    """Combina instituições brutas + detecção textual e devolve campos normalizados."""
    raw_candidates = []

    for col in ['article_institutions', 'institutions_detected_v1']:
        values = row.get(col, [])
        if isinstance(values, list):
            raw_candidates.extend(values)

    normalized_items = []

    for raw in raw_candidates:
        item = normalize_institution_value(raw)
        if item:
            normalized_items.append(item)

    # Só roda detecção textual quando necessário ou como complemento leve.
    text_for_detection = build_text_for_institution_detection(row)
    normalized_items.extend(detect_institutions_from_text(text_for_detection))

    deduped = {
        item['canonical']: item
        for item in normalized_items
        if item.get('canonical')
    }
    items = list(deduped.values())

    return {
        'institutions_raw_combined': sorted(set(clean_text(x) for x in raw_candidates if clean_text(x))),
        'institutions_normalized': sorted(item['canonical'] for item in items),
        'institution_states': sorted(set(item.get('state', '') for item in items if item.get('state'))),
        'institution_regions': sorted(set(item.get('region', '') for item in items if item.get('region'))),
        'institution_countries': sorted(set(item.get('country', '') for item in items if item.get('country'))),
        'main_institution': items[0]['canonical'] if items else '',
        'main_state': items[0].get('state', '') if items else '',
        'main_region': items[0].get('region', '') if items else '',
        'main_country': items[0].get('country', '') if items else '',
    }


## 5. Tipo de documento, campo e metodologia

In [7]:
def classify_document_type_v2(row):
    """Classificador conservador: SciELO é majoritariamente artigo."""
    title = normalize_key(row.get('article_title', ''))
    text_start = normalize_key(str(row.get('article_full_text_sample', ''))[:1500])
    combined = f'{title} {text_start}'

    if re.search(r'\bresenha\b|\bbook review\b', combined):
        return 'Resenha'

    if re.search(r'^editorial\b|\beditorial\s*:', title):
        return 'Editorial'

    if re.search(r'\bcomunicacao rapida\b|\brapid communication\b', combined):
        return 'Comunicação rápida'

    return 'Artigo'


FIELD_PATTERNS = {
    'Sociologia': ['sociologia', 'bourdieu', 'campo', 'capital simbolico', 'classe dominante', 'classes dirigentes', 'elite social', 'estratificacao', 'desigualdade', 'distincao'],
    'Ciência Política': ['ciencia politica', 'politica', 'partido', 'parlamento', 'deputado', 'senado', 'eleicao', 'eleicoes', 'democracia', 'representacao', 'elite politica', 'estado'],
    'História': ['historia', 'historico', 'seculo', 'imperio', 'republica', 'arquivo', 'memoria', 'historiografia'],
    'Direito': ['direito', 'juridico', 'judiciario', 'tribunal', 'constituicao', 'magistratura', 'justica', 'stf'],
    'Educação': ['educacao', 'ensino superior', 'universidade', 'escola', 'capital escolar', 'trajetoria escolar', 'professores'],
    'Geografia': ['geografia', 'territorio', 'espaco', 'urbano', 'cidade', 'regiao', 'regional', 'segregacao'],
    'Economia': ['economia', 'economico', 'mercado', 'empresa', 'empresarial', 'renda', 'capital economico', 'financeiro'],
    'Antropologia': ['antropologia', 'etnografia', 'cultura', 'parentesco', 'ritual', 'simbolico'],
    'Saúde': ['saude', 'medicina', 'medico', 'sanitario', 'hospital', 'saude publica'],
}

METHODOLOGY_PATTERNS = {
    'Quantitativa': ['quantitativo', 'estatistica', 'regressao', 'modelo', 'survey', 'questionario', 'base de dados', 'banco de dados', 'amostra', 'analise fatorial', 'analise de correspondencia', 'cluster', 'serie temporal', 'dados longitudinais'],
    'Qualitativa': ['qualitativo', 'entrevista', 'entrevistas', 'etnografia', 'observacao participante', 'estudo de caso', 'historia oral', 'analise de discurso', 'trajetoria', 'narrativa'],
    'Bibliográfica/Teórica': ['revisao bibliografica', 'revisao de literatura', 'ensaio teorico', 'discussao teorica', 'estado da arte', 'literatura', 'teoria'],
    'Documental': ['documental', 'arquivo', 'fontes documentais', 'documentos', 'atas', 'relatorios', 'legislacao', 'jornais'],
    'Comparativa': ['comparativo', 'comparativa', 'comparacao', 'comparada', 'entre paises', 'entre regioes', 'cross national'],
}


def classify_by_patterns(row, pattern_dict, default_value):
    text = ' '.join([
        row.get('article_title', ''),
        row.get('article_journal', ''),
        row.get('article_abstract', ''),
        ' '.join(row.get('article_keywords', []) if isinstance(row.get('article_keywords', []), list) else []),
        str(row.get('article_full_text_sample', ''))[:12000],
    ])

    text_key = normalize_key(text)
    detected = []

    for label, terms in pattern_dict.items():
        if any(normalize_key(term) in text_key for term in terms):
            detected.append(label)

    if pattern_dict is METHODOLOGY_PATTERNS and 'Quantitativa' in detected and 'Qualitativa' in detected:
        return ['Mista'] + detected

    return detected if detected else [default_value]

## 6. Leitura dos arquivos enriquecidos

In [8]:
def discover_enriched_files():
    """Procura arquivos enriquecidos em locais e padrões prováveis."""
    patterns = [
        '02_scielo_articles_*_enriched.csv',
        'enriched_*.csv',
        '*_enriched.csv',
    ]

    candidate_dirs = [
        ENRICHED_DIR,
        PROJECT_DIR / 'Scielo' / '02.data_enrich',
        PROJECT_DIR,
    ]

    found = {}

    for directory in candidate_dirs:
        if not directory.exists():
            continue

        for pattern in patterns:
            for path in directory.glob(pattern):
                if path.is_file():
                    found[path.resolve()] = path

    # fallback: busca recursiva no projeto
    if not found:
        for pattern in patterns:
            for path in PROJECT_DIR.rglob(pattern):
                if path.is_file():
                    found[path.resolve()] = path

    files = sorted(found.values())

    # Preferir arquivos por keyword. Se só houver combinado, usa combinado.
    non_combined = [
        path for path in files
        if 'combined' not in path.name.lower()
        and 'checkpoint' not in path.name.lower()
        and 'skipped' not in path.name.lower()
    ]

    files = non_combined if non_combined else files

    if KEYWORD_SLUGS is not None:
        allowed = set(KEYWORD_SLUGS)
        files = [
            path for path in files
            if get_keyword_slug_from_enriched_file(path) in allowed
        ]

    return files


enriched_files = discover_enriched_files()

print(f'Arquivos enriquecidos encontrados: {len(enriched_files)}')
for path in enriched_files:
    size_kb = path.stat().st_size / 1024
    slug = get_keyword_slug_from_enriched_file(path)
    print(f'- {path} | {size_kb:.1f} KB | slug={slug}')

Arquivos enriquecidos encontrados: 16
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/02_enriched/02_scielo_articles_burguesia_enriched.csv | 5670.5 KB | slug=burguesia
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/02_enriched/02_scielo_articles_burguesia_industrial_enriched.csv | 885.7 KB | slug=burguesia_industrial
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/02_enriched/02_scielo_articles_campo_do_poder_enriched.csv | 65289.5 KB | slug=campo_do_poder
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/02_enriched/02_scielo_articles_classe_dominante_enriched.csv | 4390.6 KB | slug=classe_dominante
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/02_enriched/02_scielo_articles_classes_dirigentes_enriched.csv | 1391.9 KB | slug=classes_dirigentes
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/02_enriched/02_scielo_articles_cl

## 7. Pipeline de normalização por arquivo

In [9]:
def normalize_enriched_df(df, keyword_slug):
    """Normaliza uma base enriquecida de uma keyword."""
    out = df.copy()

    list_cols = [
        'authors', 'article_authors', 'article_institutions', 'article_keywords',
        'article_references', 'institutions_detected_v1', 'states_detected', 'regions_detected'
    ]

    for col in list_cols:
        if col in out.columns:
            out[col] = out[col].apply(safe_literal_list)

    text_cols = [
        'title', 'journal', 'year', 'doi', 'abstract',
        'article_title', 'article_journal', 'article_publication_date', 'article_doi',
        'article_language', 'article_publisher', 'article_abstract',
        'article_full_text_sample', 'article_scrape_error', 'link', 'source_text',
        'search_keyword', 'search_group', 'search_keyword_slug'
    ]

    for col in text_cols:
        if col in out.columns:
            out[col] = out[col].apply(clean_text)

    # Colunas canônicas com fallback.
    out['article_title'] = out.apply(lambda r: r.get('article_title', '') or r.get('title', ''), axis=1)
    out['article_journal'] = out.apply(lambda r: r.get('article_journal', '') or r.get('journal', ''), axis=1)
    out['article_doi'] = out.apply(lambda r: r.get('article_doi', '') or r.get('doi', ''), axis=1)
    out['article_abstract'] = out.apply(lambda r: r.get('article_abstract', '') or r.get('abstract', ''), axis=1)

    if 'article_authors' not in out.columns:
        out['article_authors'] = [[] for _ in range(len(out))]

    out['article_authors'] = out.apply(
        lambda r: r['article_authors'] if isinstance(r['article_authors'], list) and len(r['article_authors']) > 0 else r.get('authors', []),
        axis=1,
    )

    out['article_year'] = out['article_publication_date'].apply(extract_year) if 'article_publication_date' in out.columns else pd.NA
    fallback_year = out['year'].apply(extract_year) if 'year' in out.columns else pd.Series([pd.NA] * len(out), index=out.index)
    out['article_year'] = out['article_year'].fillna(fallback_year)
    out['article_year'] = pd.to_numeric(out['article_year'], errors='coerce').astype('Int64')

    # Filtro temporal do projeto.
    out = out[out['article_year'].between(START_YEAR, END_YEAR, inclusive='both')].copy()

    if out.empty:
        return out

    out['document_type_v2'] = out.apply(classify_document_type_v2, axis=1)
    out['is_core_research_article_v2'] = out['document_type_v2'].eq('Artigo')

    # Normalização institucional.
    # Usamos list comprehension com tqdm em vez de DataFrame.apply para ter progresso visível
    # e evitar overhead adicional do pandas em linhas com textos grandes.
    institution_rows = pd.DataFrame([
        normalize_institutions_for_row(row)
        for row in tqdm(
            out.to_dict('records'),
            total=len(out),
            desc=f'Normalizando instituições: {keyword_slug}',
            leave=False,
        )
    ])
    out = pd.concat([out.reset_index(drop=True), institution_rows.reset_index(drop=True)], axis=1)

    # Compatibilidade com nomes antigos.
    out['institutions_detected_v1'] = out['institutions_normalized']
    out['states_detected'] = out['institution_states']
    out['regions_detected'] = out['institution_regions']

    out['fields_detected'] = out.apply(lambda r: classify_by_patterns(r, FIELD_PATTERNS, 'Indefinido'), axis=1)
    out['methodologies_detected'] = out.apply(lambda r: classify_by_patterns(r, METHODOLOGY_PATTERNS, 'Não identificada'), axis=1)

    if 'search_keyword_slug' not in out.columns:
        out['search_keyword_slug'] = keyword_slug

    out['search_keyword_slug'] = out['search_keyword_slug'].replace('', keyword_slug).fillna(keyword_slug)

    if 'search_keyword' not in out.columns:
        out['search_keyword'] = keyword_slug.replace('_', ' ')

    out['search_keyword'] = out['search_keyword'].replace('', keyword_slug.replace('_', ' ')).fillna(keyword_slug.replace('_', ' '))

    if 'search_group' not in out.columns:
        out['search_group'] = ''

    out['source_file'] = f'02_scielo_articles_{keyword_slug}_enriched.csv'

    return out


def process_enriched_file(path):
    keyword_slug = get_keyword_slug_from_enriched_file(path)

    df, error = load_csv_safe(
        path,
        required_any=['link', 'article_title', 'title', 'article_doi', 'doi']
    )

    if df is None:
        print(f'[SKIP] {path.name}: {error}')
        return None, {
            'file': str(path),
            'keyword_slug': keyword_slug,
            'reason': error,
            'stage': '03_normalize',
        }

    clean_df = normalize_enriched_df(df, keyword_slug=keyword_slug)

    if clean_df.empty:
        print(f'[SKIP] {path.name}: sem linhas após filtro temporal/normalização')
        return None, {
            'file': str(path),
            'keyword_slug': keyword_slug,
            'reason': 'sem linhas após filtro temporal/normalização',
            'stage': '03_normalize',
        }

    output_path = CLEAN_DIR / f'03_corpus_{keyword_slug}_scielo_clean.csv'
    sheets_path = CLEAN_SHEETS_DIR / f'03_corpus_{keyword_slug}_scielo_clean_sheets.csv'

    serialize_cols = [
        'authors', 'article_authors', 'article_institutions', 'article_keywords', 'article_references',
        'institutions_raw_combined', 'institutions_normalized', 'institutions_detected_v1',
        'institution_states', 'institution_regions', 'institution_countries',
        'states_detected', 'regions_detected', 'fields_detected', 'methodologies_detected'
    ]

    serialize_list_columns(clean_df, serialize_cols).to_csv(output_path, index=False, encoding='utf-8-sig')
    create_google_sheets_friendly_df(clean_df).to_csv(sheets_path, index=False, encoding='utf-8-sig')

    print(f'{keyword_slug}: {len(clean_df)} linhas salvas')
    print(f'  técnico: {output_path}')
    print(f'  sheets : {sheets_path}')

    return clean_df, None

In [10]:
all_clean = []
skipped_files = []

for path in enriched_files:
    clean_df, skipped = process_enriched_file(path)

    if skipped:
        skipped_files.append(skipped)
        continue

    all_clean.append(clean_df)

combined_clean = pd.concat(all_clean, ignore_index=True) if all_clean else pd.DataFrame()

serialize_cols = [
    'authors', 'article_authors', 'article_institutions', 'article_keywords', 'article_references',
    'institutions_raw_combined', 'institutions_normalized', 'institutions_detected_v1',
    'institution_states', 'institution_regions', 'institution_countries',
    'states_detected', 'regions_detected', 'fields_detected', 'methodologies_detected'
]

combined_path = CLEAN_DIR / '03_corpus_scielo_clean_combined.csv'
combined_sheets_path = CLEAN_DIR / '03_corpus_scielo_clean_combined_sheets.csv'

if not combined_clean.empty:
    serialize_list_columns(combined_clean, serialize_cols).to_csv(
        combined_path,
        index=False,
        encoding='utf-8-sig',
    )

    create_google_sheets_friendly_df(combined_clean).to_csv(
        combined_sheets_path,
        index=False,
        encoding='utf-8-sig',
    )

    print('Base combinada salva:')
    print('-', combined_path)
    print('-', combined_sheets_path)
else:
    print('Nenhuma base limpa foi criada.')

if skipped_files:
    skipped_path = LOG_DIR / 'skipped_normalize_files.csv'
    pd.DataFrame(skipped_files).to_csv(skipped_path, index=False, encoding='utf-8-sig')
    print(f'Arquivos pulados registrados em: {skipped_path}')

print(combined_clean.shape)
combined_clean.head()

/var/folders/rv/2hqg76v54gj0hlbj67c5ftk80000gn/T/ipykernel_7708/3933287737.py:7: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  value = BeautifulSoup(value, 'html.parser').get_text(' ', strip=True)
/var/folders/rv/2hqg7

burguesia: 80 linhas salvas
  técnico: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_burguesia_scielo_clean.csv
  sheets : /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_burguesia_scielo_clean_sheets.csv


burguesia_industrial: 13 linhas salvas
  técnico: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_burguesia_industrial_scielo_clean.csv
  sheets : /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_burguesia_industrial_scielo_clean_sheets.csv


/var/folders/rv/2hqg76v54gj0hlbj67c5ftk80000gn/T/ipykernel_7708/3933287737.py:7: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  value = BeautifulSoup(value, 'html.parser').get_text(' ', strip=True)
/var/

campo_do_poder: 874 linhas salvas
  técnico: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_campo_do_poder_scielo_clean.csv
  sheets : /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_campo_do_poder_scielo_clean_sheets.csv


...

grupos_dirigentes: 39 linhas salvas
  técnico: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_grupos_dirigentes_scielo_clean.csv
  sheets : /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_grupos_dirigentes_scielo_clean_sheets.csv


quem_governa: 5 linhas salvas
  técnico: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_quem_governa_scielo_clean.csv
  sheets : /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_quem_governa_scielo_clean_sheets.csv


/var/folders/rv/2hqg76v54gj0hlbj67c5ftk80000gn/T/ipykernel_7708/3933287737.py:7: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  value = BeautifulSoup(value, 'html.parser').get_text(' ', strip=True)


sociologia_das_elites: 58 linhas salvas
  técnico: /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_sociologia_das_elites_scielo_clean.csv
  sheets : /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_sociologia_das_elites_scielo_clean_sheets.csv
Base combinada salva:
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_scielo_clean_combined.csv
- /Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos/data/03_clean/03_corpus_scielo_clean_combined_sheets.csv
(2662, 50)


,scielo_id,title,link,authors,journal,year,source_text,doi,abstract,total_hits_on_search,...,main_state,main_region,main_country,institutions_detected_v1,states_detected,regions_detected,fields_detected,methodologies_detected,search_group,source_file
0,S0011-52582025000100209-scl,Antonio Gramsci em Refração: os Usos de Fernan...,http://www.scielo.br/scielo.php?script=sci_art...,"[Mussi, Daniela, Herscovici, Nicole]",Dados,2025,"Dados , mar 2025, Volume 68 N. 1 elocation e20...",10.1590/dados.2025.68.1.351,Abstract This article aims to investigate a un...,80,...,RJ,Sudeste,Brasil,"[Universidade Federal do Rio de Janeiro, Unive...","[RJ, SP]",[Sudeste],"[Ciência Política, História, Geografia, Econom...","[Qualitativa, Bibliográfica/Teórica, Documental]",,02_scielo_articles_burguesia_enriched.csv
1,S1517-106X2025000300412-scl,“Este lixo do engenho humano”: um sentido atua...,http://www.scielo.br/scielo.php?script=sci_art...,"[Abbati, Orietta]",Alea: Estudos Neolatinos,2025,"Alea: Estudos Neolatinos , 2025, Volume 27 N. ...",10.1590/1517-106x/2025e69398,Abstract The article reflects on the short sto...,80,...,,,,"[Universidade Federal do Rio de Janeiro, schoo...",[RJ],[Sudeste],"[Sociologia, Ciência Política, História, Educa...",[Qualitativa],,02_scielo_articles_burguesia_enriched.csv
2,S1517-106X2025000300410-scl,O primo Basílio: estética e desnudamento da bu...,http://www.scielo.br/scielo.php?script=sci_art...,"[Silva, Vera Lopes da]",Alea: Estudos Neolatinos,2025,"Alea: Estudos Neolatinos , 2025, Volume 27 N. ...",10.1590/1517-106x/2025e69394,Resumen Este estudio se dedica a la novela El ...,80,...,,,,"[Universidade Federal do Rio de Janeiro, schoo...",[RJ],[Sudeste],"[Ciência Política, História, Direito, Geografi...","[Mista, Quantitativa, Qualitativa, Bibliográfi...",,02_scielo_articles_burguesia_enriched.csv
3,S0034-83092025000100303-scl,HORTICULTURA ORNAMENTAL NO SÉCULO XIX: UMA LIN...,http://www.scielo.br/scielo.php?script=sci_art...,"[Far, Alessandra El]",Revista de História (São Paulo),2025,"Revista de História (São Paulo) , 2025, N. 184...",10.11606/issn.2316-9141.rh.2025.225637,Resumo Este artigo tem o propósito de elucidar...,80,...,,,,"[Universidade de São Paulo, school Universidad...",[SP],[Sudeste],"[Sociologia, Ciência Política, História, Geogr...",[Qualitativa],,02_scielo_articles_burguesia_enriched.csv
4,S0104-06182025000100302-scl,Renta agraria y derechos de exportación durant...,http://www.scielo.br/scielo.php?script=sci_art...,"[Bernado, Rolando Garcia, Amoretti, Leandro]",Economia e Sociedade,2025,"Economia e Sociedade , 2025, Volume 34 N. 1 el...",10.1590/1982-3533.2025v34n1.267170,Resumo A administração Macri (2015-2019) veio ...,80,...,,,,"[Universidade Estadual de Campinas, school Inv...",[SP],[Sudeste],"[Ciência Política, História, Direito, Economia...",[Não identificada],,02_scielo_articles_burguesia_enriched.csv


## 8. Diagnóstico da normalização

In [11]:
if combined_clean.empty:
    print('Nenhuma base limpa criada.')
else:
    diagnostic_clean = (
        combined_clean
        .assign(
            has_doi=lambda x: x['article_doi'].fillna('').ne(''),
            has_abstract=lambda x: x['article_abstract'].fillna('').ne(''),
            has_keywords=lambda x: x['article_keywords'].apply(lambda y: isinstance(y, list) and len(y) > 0),
            has_references=lambda x: x['article_references'].apply(lambda y: isinstance(y, list) and len(y) > 0),
            has_institution=lambda x: x['institutions_normalized'].apply(lambda y: isinstance(y, list) and len(y) > 0),
            has_state=lambda x: x['institution_states'].apply(lambda y: isinstance(y, list) and len(y) > 0),
        )
        .groupby('search_keyword', as_index=False)
        .agg(
            records=('article_title', 'count'),
            core_articles=('is_core_research_article_v2', 'sum'),
            unique_titles=('article_title', 'nunique'),
            dois=('has_doi', 'sum'),
            abstracts=('has_abstract', 'sum'),
            keywords=('has_keywords', 'sum'),
            references=('has_references', 'sum'),
            institutions=('has_institution', 'sum'),
            states=('has_state', 'sum'),
            min_year=('article_year', 'min'),
            max_year=('article_year', 'max'),
        )
        .sort_values('records', ascending=False)
    )

    diagnostic_clean.to_csv(CLEAN_DIR / 'diagnostic_clean_by_keyword.csv', index=False, encoding='utf-8-sig')
    display(diagnostic_clean)

,search_keyword,records,core_articles,unique_titles,dois,abstracts,keywords,references,institutions,states,min_year,max_year
2,campo do poder,874,874,872,866,874,856,872,858,756,1978,2025
10,elites,632,629,631,630,616,592,630,612,540,1983,2025
11,grupo de poder,540,540,540,537,540,532,538,527,397,1977,2025
9,elite política,202,202,202,200,201,194,202,196,174,1996,2025
0,burguesia,80,79,80,79,79,76,80,78,68,1982,2025
5,classes dominantes,64,64,64,64,64,63,64,63,48,1979,2025
3,classe dominante,62,62,62,62,62,59,62,61,52,1986,2025
15,sociologia das elites,58,58,58,58,57,54,58,57,57,2000,2025
6,elite econômica,47,47,47,47,47,45,47,46,38,1998,2024
13,grupos dirigentes,39,39,39,37,38,35,39,38,37,1976,2025
